In [8]:
import os
import mesa
import time
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load your working API key from the .env file
load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

# Gemini 2.5 Flash Pricing (per 1 Million tokens)
# Prompts: $0.075 / 1M | Responses: $0.30 / 1M
PROMPT_COST_PER_TOKEN = 0.075 / 1_000_000
RESPONSE_COST_PER_TOKEN = 0.30 / 1_000_000

class EconomicAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)

    def step(self):
        prompt = f"Agent {self.unique_id} requires instructions. Respond STRICTLY with JSON: {{'action': 'wait'}}"
        
        try:
            # Call the API
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json")
            )
            
            # --- THE CORE UPGRADE: TOKEN EXTRACTION ---
            input_tokens = response.usage_metadata.prompt_token_count
            output_tokens = response.usage_metadata.candidates_token_count
            
            # Calculate the exact fractional USD cost
            step_cost = (input_tokens * PROMPT_COST_PER_TOKEN) + (output_tokens * RESPONSE_COST_PER_TOKEN)
            
            # Send the bill to the central Mesa Model
            self.model.log_expense(self.unique_id, input_tokens, output_tokens, step_cost)
            
        except Exception as e:
            print(f"Agent {self.unique_id} failed: {e}")
            
        # 4-Second throttle to protect your free tier
        time.sleep(4)

class EconomicModel(mesa.Model):
    def __init__(self, n_agents):
        super().__init__()
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_usd_cost = 0.0
        
        for i in range(n_agents):
            agent = EconomicAgent(self)
            
    def log_expense(self, agent_id, input_tokens, output_tokens, cost):
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_usd_cost += cost
        print(f"[BILLING] Agent {agent_id} consumed {input_tokens} In / {output_tokens} Out tokens. Cost: ${cost:.6f}")
        
    def step(self):
        self.agents.shuffle_do("step")
        
    def generate_financial_report(self):
        print("\n" + "="*50)
        print("💰 MESA-LLM SIMULATION FINANCIAL REPORT")
        print("="*50)
        print(f"Total Prompt Tokens:     {self.total_input_tokens:,}")
        print(f"Total Candidate Tokens:  {self.total_output_tokens:,}")
        print(f"Total Tokens Processed:  {self.total_input_tokens + self.total_output_tokens:,}")
        print("-" * 50)
        print(f"TOTAL SIMULATION COST:   ${self.total_usd_cost:.6f} USD")
        print("="*50)

In [9]:
finance_model = EconomicModel(3)

print("Starting simulation. Tracking token burn rate...")
for _ in range(2):
    finance_model.step()
finance_model.generate_financial_report()

Starting simulation. Tracking token burn rate...
[BILLING] Agent 2 consumed 19 In / 11 Out tokens. Cost: $0.000005
[BILLING] Agent 3 consumed 19 In / 11 Out tokens. Cost: $0.000005
[BILLING] Agent 1 consumed 19 In / 11 Out tokens. Cost: $0.000005
[BILLING] Agent 1 consumed 19 In / 11 Out tokens. Cost: $0.000005
[BILLING] Agent 2 consumed 19 In / 11 Out tokens. Cost: $0.000005
[BILLING] Agent 3 consumed 19 In / 11 Out tokens. Cost: $0.000005

💰 MESA-LLM SIMULATION FINANCIAL REPORT
Total Prompt Tokens:     114
Total Candidate Tokens:  66
Total Tokens Processed:  180
--------------------------------------------------
TOTAL SIMULATION COST:   $0.000028 USD
